In [4]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import warnings
warnings.filterwarnings("ignore")


#### **Tool Creation**

In [7]:
# TOOL - 1 [News Search Tool]

from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool

#earch_tool = DuckDuckGoSearchRun(description="This is a tool to search the web for news")

#pip install -U ddgs

## as its internet interaction so SSL issue comes thats why using dummy tool instead
# TOOL - 1 [Dummy News search tool]

@tool
def news_search_tool(query: str) -> str:
    """Search for news articles"""
    
    return f"""
    Dummy News Results for: {query}

    1. AI continues to grow in enterprise applications.
    2. Open-source LLMs gain popularity among developers.
    3. Companies increasingly adopt local AI deployments.
    """




In [8]:
# TOOL - 2 [Wikipedia Search Tool]
# pip install wikipedia
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.tools import tool


wikipedia_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(),description="This is a tool to search Wikipedia")


# TOOL - 2 [Dummy Wikipedia Tool]

@tool
def wikipedia_tool(query: str) -> str:
    """Search Wikipedia articles"""

    dummy_data = {
        "india": "India is a country in South Asia.",
        "python": "Python is a high-level programming language.",
        "mistral": "Mistral is a family of open-source language models."
    }

    return dummy_data.get(
        query.lower(),
        f"No Wikipedia article found for '{query}'"
    )


In [9]:
# TOOL - 3 [Custom Enterprise Tool]

# pip install  langchain
from langchain.tools import tool

@tool
def enterprise_tool(query:str)-> str:

    """This is a tool to send emails to employees"""
    
    return "Email Sent"

In [10]:
ToolKit = [
    news_search_tool,
    wikipedia_tool,
    enterprise_tool
]

ToolKit

[StructuredTool(name='news_search_tool', description='Search for news articles', args_schema=<class 'langchain_core.utils.pydantic.news_search_tool'>, func=<function news_search_tool at 0x0000028ACEC57C40>),
 StructuredTool(name='wikipedia_tool', description='Search Wikipedia articles', args_schema=<class 'langchain_core.utils.pydantic.wikipedia_tool'>, func=<function wikipedia_tool at 0x0000028ACEC57B00>),
 StructuredTool(name='enterprise_tool', description='This is a tool to send emails to employees', args_schema=<class 'langchain_core.utils.pydantic.enterprise_tool'>, func=<function enterprise_tool at 0x0000028ACEC568E0>)]

#### **ReAct Agent Creation**

In [45]:
#from langchain_community.chat_models import ChatOllama
#from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent
import requests


# model = ChatOllama(
#     model="mistral:latest",
#     temperature=0.7
# )

def llm(prompt: str) -> str:
    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": "mistral:latest",
            "messages": [{"role": "user", "content": prompt}],
            "stream": False
        }
    )
    return response.json()["message"]["content"]




# 3. TOOL ROUTER (FIXED)
# =========================

def agent(user_input: str):
    text = user_input.lower()

    if "news" in text:
        return news_search_tool.invoke({"query": user_input})

    if "wiki" in text or "what is" in text:
        return wikipedia_tool.invoke({"query": user_input})

    if "email" in text or "send" in text:
        return enterprise_tool.invoke({"query": user_input})

    return llm(user_input)

# # agent create

# from langgraph.prebuilt import create_react_agent

# agent = create_react_agent(
#     model=model,
#     tools=ToolKit
# )

print("Agent ready")
print("Agent created successfully")

Agent ready
Agent created successfully


#### Test agents

In [46]:
# response = agent.invoke(
#     {
#         "messages": [
#             ("user", "Search news about AI and send email summary")
#         ]
#     }
# )

# print(response)

print("=== TEST 1 ===")
print(agent("Search news about AI"))

print("\n=== TEST 2 ===")
print(agent("What is Python?"))

print("\n=== TEST 3 ===")
print(agent("Send email to team about meeting"))

print("\n=== TEST 4 ===")
print(agent("Explain machine learning"))

=== TEST 1 ===

    [NEWS]
    Query: Search news about AI

    - AI adoption is increasing
    - Offline LLMs are trending
    - Enterprises prefer local AI
    

=== TEST 2 ===
No result for What is Python?

=== TEST 3 ===
Email sent: Send email to team about meeting

=== TEST 4 ===
 Machine Learning is a subset of artificial intelligence (AI) that provides systems the ability to automatically learn and improve from experience without being explicitly programmed. It focuses on the development of computer programs that can access data and use it to learn for themselves.

Machine learning algorithms are designed to "learn" patterns in data, and then make decisions or predictions based on the patterns learned. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning.

1. Supervised Learning: In this type, the algorithm is trained on a labeled dataset where the correct output (label) is provided for each example in the input d

In [ ]:
example_query = "Give me the latest news about the stock market"

events = agent.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values"
)

for event in events:
    # safely print last message
    if "messages" in event:
        msg = event["messages"][-1]

        # LangGraph message object handling
        if hasattr(msg, "pretty_print"):
            msg.pretty_print()
        else:
            print(msg)